# **Environment Verification and Model Setup**
---
I used this notebook to verify that the Python environment is correctly installed and that all dependencies are available before running any detection code. I also downloaded the YOLOv8 pretrained weights for the three model variants I will compare: nano, small, and medium.

In [1]:
# ── Step 1: Verify Python version ────────────────────────────────────────────
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 9), "Python 3.9+ is required."
print("✓ Python version OK")

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
✓ Python version OK


In [ ]:
# ── Step 2: Import all core packages and check they load correctly ────────────
# I imported everything here so that any missing package produces a clear error now, not in a later notebook.

import torch
import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import ultralytics
import psutil
import yaml
import streamlit  # just importing to verify it's installed

print(f"torch        : {torch.__version__}")
print(f"ultralytics  : {ultralytics.__version__}")
print(f"opencv       : {cv2.__version__}")
print(f"numpy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")
print(f"seaborn      : {sns.__version__}")
print("\n✓ All packages imported successfully")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Peter\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
torch        : 2.12.0+cpu
ultralytics  : 8.4.66
opencv       : 4.13.0
numpy        : 2.2.6
pandas       : 2.2.2
matplotlib   : 3.8.4
seaborn      : 0.13.2

✓ All packages imported successfully


In [ ]:
# ── Step 3: Check GPU / CPU availability ─────────────────────────────────────
# GPU is not required — the project runs on CPU. But for reproducibility, if CUDA is available, inference will be significantly faster.

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version : {torch.version.cuda}")
    print(f"GPU memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("GPU not available — running on CPU.")
    print("Expected CPU inference speeds: nano ~20 FPS, small ~12 FPS, medium ~6 FPS.")

# Report system RAM
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"\nSystem RAM   : {ram_gb:.1f} GB")
print(f"Device to use: {device}")

GPU not available — running on CPU.
Expected CPU inference speeds: nano ~20 FPS, small ~12 FPS, medium ~6 FPS.

System RAM   : 15.8 GB
Device to use: cpu


In [ ]:
# ── Step 4: Set up project paths ─────────────────────────────────────────────
# I added the src/ directory to sys.path so I can import the detection_system package from any notebook without installing it as a package.

import sys
from pathlib import Path

# This cell assumes the notebook is inside the notebooks/ directory.
# PROJECT_ROOT is the parent of notebooks/.
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root : {PROJECT_ROOT}")
print(f"src/ added to sys.path: {SRC_DIR.exists()}")

# Verify the package loads
from detection_system import load_model, get_coco_classes
print("✓ detection_system package imported successfully")

Project root : c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection
src/ added to sys.path: True
✓ detection_system package imported successfully


In [5]:
# ── Step 5: Load project config ───────────────────────────────────────────────
from detection_system.utils import load_config

cfg = load_config("config.yaml")
print("Config loaded:")
for section, values in cfg.items():
    print(f"  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"    {k}: {v}")
    else:
        print(f"    {values}")

Config loaded:
  [project]
    name: Real-Time Object Detection and Counting
    author: P. O. Adepoju
    version: 1.0.0
  [model]
    default_variant: small
    confidence_threshold: 0.25
    iou_threshold: 0.45
    target_classes: None
  [benchmark]
    variants_to_compare: ['nano', 'small', 'medium']
    n_warmup_frames: 10
    n_benchmark_frames: 100
    frame_width: 640
    frame_height: 480
  [video]
    default_sample: sample_traffic.mp4
    output_fps: 25
  [counting]
    top_n_classes: 5
  [figures]
    dpi: 300
    style: seaborn-v0_8-whitegrid
    palette: colorblind
    figsize_default: [10, 6]
    figsize_wide: [14, 5]
  [seed]
    42


In [ ]:
# ── Step 6: Downloading the YOLOv8 weights ──────────────────────────────────────────

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

variants_to_download = cfg["benchmark"]["variants_to_compare"]
print(f"Downloading variants: {variants_to_download}")
print(f"Saving to: {MODELS_DIR}\n")

models = {}
for variant in variants_to_download:
    print(f"--- Loading YOLOv8 {variant} ---")
    model = load_model(variant, models_dir=MODELS_DIR)
    models[variant] = model
    print()

Saving to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models

--- Loading YOLOv8 nano ---
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 NANO | Classes: 80

--- Loading YOLOv8 small ---
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 SMALL | Classes: 80

--- Loading YOLOv8 medium ---
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 MEDIUM | Classes: 80



In [7]:
# ── Step 7: Verify weights on disk ───────────────────────────────────────────
import os

print("Model weight files in models/:")
print(f"{'Filename':<20} {'Size (MB)':>12}")
print("-" * 34)

for pt_file in sorted(MODELS_DIR.glob("*.pt")):
    size_mb = pt_file.stat().st_size / (1024 ** 2)
    print(f"{pt_file.name:<20} {size_mb:>11.1f}")

print("\n✓ Weights verified on disk")

Model weight files in models/:
Filename                Size (MB)
----------------------------------

✓ Weights verified on disk


In [ ]:
# ── Step 8: Quick smoke test ──────────────
# I created a blank 640×480 frame and run the small model on it. A blank frame should produce zero detections — this tests the pipeline without needing a real image.

from detection_system.inference import detect_frame

blank_frame = np.zeros((480, 640, 3), dtype=np.uint8)
detections, latency = detect_frame(models["small"], blank_frame, confidence=0.25)

print(f"Blank frame smoke test:")
print(f"  Detections  : {len(detections)} (expected: 0)")
print(f"  Latency     : {latency:.1f} ms")
print(f"  FPS equiv.  : {1000/latency:.1f}")

# Interpretation: a blank frame should produce no detections.
# If detections > 0 here, the confidence threshold may be too low.
assert len(detections) == 0, "Unexpected detections on a blank frame!"
print("\n✓ Smoke test passed — pipeline is working correctly")

Blank frame smoke test:
  Detections  : 0 (expected: 0)
  Latency     : 600.1 ms
  FPS equiv.  : 1.7

✓ Smoke test passed — pipeline is working correctly


In [9]:
# ── Step 9: Inspect COCO class names ─────────────────────────────────────────
from detection_system.loader import get_coco_classes

coco_classes = get_coco_classes()
print(f"Total COCO classes: {len(coco_classes)}")
print("\nFirst 10 classes:")
for idx, name in enumerate(coco_classes[:10]):
    print(f"  [{idx:2d}] {name}")

print("\nVehicle-related classes (useful for traffic videos):")
vehicle_keywords = ["car", "truck", "bus", "motorcycle", "bicycle", "train", "boat"]
for idx, name in enumerate(coco_classes):
    if any(kw in name for kw in vehicle_keywords):
        print(f"  [{idx:2d}] {name}")

Total COCO classes: 80

First 10 classes:
  [ 0] person
  [ 1] bicycle
  [ 2] car
  [ 3] motorcycle
  [ 4] airplane
  [ 5] bus
  [ 6] train
  [ 7] truck
  [ 8] boat
  [ 9] traffic light

Vehicle-related classes (useful for traffic videos):
  [ 1] bicycle
  [ 2] car
  [ 3] motorcycle
  [ 5] bus
  [ 6] train
  [ 7] truck
  [ 8] boat
  [51] carrot
